# LeetCode 686 — Repeated String Match bằng Python

Link đề: https://leetcode.com/problems/repeated-string-match/ · Độ khó: **Medium** · Chủ đề: String, String Matching, KMP

Skeleton LeetCode:

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp chuỗi `a` tối thiểu bao nhiêu lần để chuỗi `b` trở thành **chuỗi con liền nhau (substring)** của kết quả; không thể thì trả `-1`.
2. Nắm các công cụ Python: nhân chuỗi `a * k`, toán tử `in` (kiểm tra chuỗi con), `str.find`, **chia lấy trần** (`-(-x // y)`), chỉ số vòng bằng `%`, và hàm **tiền tố–hậu tố (prefix function)** của thuật toán KMP.
3. Đi từ **thử lặp `1, 2, 3, ...` đến khi thấy** → **chặn trên số lần cần thử (`q` hoặc `q + 1`)** → **không dựng chuỗi, chỉ dùng chỉ số vòng `% len(a)`** → **KMP trên chuỗi lặp "ảo"** (tuyến tính).
4. Hiểu mấu chốt: nếu `b` xuất hiện thì có thể "trượt" để nó **bắt đầu trong bản sao đầu tiên**; khi đó số lần lặp chỉ phụ thuộc vị trí bắt đầu `s < len(a)` và `len(b)` ⇒ chỉ cần thử `ceil(len(b)/len(a))` hoặc **cộng thêm một**.
5. Tránh các lỗi kinh điển: dùng chia nguyên thay vì chia trần, quên bản "dư thêm một lần" cho trường hợp `b` vắt qua ranh giới, vòng lặp không có điều kiện dừng (lặp vô hạn), và kết luận "đủ chữ cái ⇒ có đáp án".

## 2. Đề bài & ví dụ

Cho hai chuỗi chữ thường `a` và `b`. Trả về **số lần tối thiểu** phải lặp `a` sao cho `b` là **chuỗi con** của chuỗi sau khi lặp. Nếu không thể, trả `-1`.

Lưu ý: `"abc"` lặp 0 lần là `""`, 1 lần là `"abc"`, 2 lần là `"abcabc"`.

**Ví dụ 1:** `a = "abcd"`, `b = "cdabcdab"` ⇒ `3`.
Giải thích: lặp 3 lần được `"abcdabcdabcd"`, trong đó `"ab` **`cdabcdab`** `cd"` chứa `b`.

**Ví dụ 2:** `a = "a"`, `b = "aa"` ⇒ `2`.

**Ràng buộc:**

- `1 <= a.length, b.length <= 10^4`
- `a`, `b` gồm chữ cái thường tiếng Anh.

Cell dưới chạy thử hai ví dụ bằng cách lặp thủ công (chưa phải thuật toán), để thấy đề đang hỏi gì.

In [1]:
import random
import time
from math import ceil
from typing import List

examples = [("abcd", "cdabcdab", 3), ("a", "aa", 2)]
for a, b, expected in examples:
    for k in range(1, 5):
        rep = a * k
        print(f"a={a!r} lặp {k} lần = {rep!r}  chứa {b!r}? {b in rep}")
    print("-> mong đợi", expected)

a='abcd' lặp 1 lần = 'abcd'  chứa 'cdabcdab'? False
a='abcd' lặp 2 lần = 'abcdabcd'  chứa 'cdabcdab'? False
a='abcd' lặp 3 lần = 'abcdabcdabcd'  chứa 'cdabcdab'? True
a='abcd' lặp 4 lần = 'abcdabcdabcdabcd'  chứa 'cdabcdab'? True
-> mong đợi 3
a='a' lặp 1 lần = 'a'  chứa 'aa'? False
a='a' lặp 2 lần = 'aa'  chứa 'aa'? True
a='a' lặp 3 lần = 'aaa'  chứa 'aa'? True
a='a' lặp 4 lần = 'aaaa'  chứa 'aa'? True
-> mong đợi 2


## 3. Kiến thức nền

### 3.1. Nhân chuỗi và toán tử `in`

- `a * k` ghép `k` bản của `a` liên tiếp (`k = 0` cho chuỗi rỗng).
- `b in s` là `True` nếu `b` là **chuỗi con liền nhau** của `s` (khác với **dãy con** có thể bỏ ký tự).
- `s.find(b)` trả **chỉ số bắt đầu** đầu tiên của `b` trong `s`, hoặc `-1` nếu không có.

In [2]:
print("ab" * 3, repr("ab" * 0))
print("bca" in "abcabc", "ac" in "abcabc")      # True False
print("abcabc".find("ca"), "abcabc".find("zz"))  # 2 -1

ababab ''
True False
2 -1


### 3.2. Chia lấy trần (ceiling division)

`7 // 3 = 2` là chia **lấy sàn** (làm tròn xuống). Ta cần `ceil(7 / 3) = 3` (làm tròn lên). Hai cách thuần số nguyên, tránh số thực:

- `-(-x // y)`: đổi dấu, chia sàn, đổi dấu lại.
- `(x + y - 1) // y`.

(`math.ceil(x / y)` cũng được với số nhỏ.)

In [3]:
x, y = 7, 3
print(x // y, -(-x // y), (x + y - 1) // y, ceil(x / y))     # 2 3 3 3
print(-(-6 // 3), -(-1 // 3), -(-9 // 4))                    # 2 1 3

2 3 3 3
2 1 3


### 3.3. Chỉ số vòng bằng `%`

Chuỗi `a` lặp vô hạn có ký tự thứ `p` (đếm từ 0) bằng `a[p % len(a)]`. Nhờ vậy ta **không cần dựng** chuỗi lặp, chỉ cần một chỉ số `p` tăng dần.

In [4]:
a = "abc"
print("".join(a[p % len(a)] for p in range(8)))     # abcabcab
print(("abc" * 3)[:8])                              # cùng kết quả

abcabcab
abcabcab


### 3.4. Hàm tiền tố–hậu tố (prefix function / mảng `lps`) của KMP

Với chuỗi `b` độ dài `m`, `lps[i]` = **độ dài của tiền tố dài nhất của `b[0..i]` đồng thời là hậu tố của `b[0..i]`** (không tính cả chuỗi đó). Ví dụ `b = "abab"`: `lps = [0, 0, 1, 2]` (tại `i = 3`, tiền tố `"ab"` cũng là hậu tố `"ab"`).

Ý nghĩa: khi đang so khớp `b` với văn bản và **gặp ký tự lệch** ở vị trí `j` của `b`, ta không cần quay lại từ đầu — ta "lùi" `j` về `lps[j-1]` (phần đã khớp vẫn còn có ích). Nhờ đó tổng thời gian tuyến tính `O(n + m)`.

In [5]:
def prefix_function(b):
    lps = [0] * len(b)
    k = 0                                  # độ dài tiền tố-hậu tố hiện tại
    for i in range(1, len(b)):
        while k > 0 and b[i] != b[k]:
            k = lps[k - 1]                 # lùi
        if b[i] == b[k]:
            k += 1
        lps[i] = k
    return lps


print(prefix_function("abab"))          # [0, 0, 1, 2]
print(prefix_function("aaaa"))          # [0, 1, 2, 3]
print(prefix_function("cdabcdab"))      # [0, 0, 0, 0, 1, 2, 3, 4]

[0, 0, 1, 2]
[0, 1, 2, 3]
[0, 0, 0, 0, 1, 2, 3, 4]


### 3.5. Tập ký tự `set` và vì sao chưa đủ

`set(b) <= set(a)` (mọi chữ cái của `b` đều có trong `a`) là **điều kiện cần** để có đáp án, nhưng **chưa đủ**: thứ tự các chữ cái cũng phải khớp. Ví dụ `a = "abc"`, `b = "ac"`: chữ `a`, `c` đều có trong `a`, nhưng `"ac"` không xuất hiện liền nhau trong `"abcabc..."`.

In [6]:
print(set("ac") <= set("abc"))                                   # True (điều kiện cần)
print(any("ac" in "abc" * k for k in range(1, 6)))               # False -> vẫn không có đáp án
print(set("abz") <= set("abc"))                                  # False -> chắc chắn -1

True
False
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `a = "abcd"` (độ dài 4), `b = "cdabcdab"` (độ dài 8).

- Lặp 1 lần: `"abcd"` — dài 4, ngắn hơn `b` ⇒ chắc chắn không chứa.
- Lặp 2 lần: `"abcdabcd"` — dài 8 đúng bằng `b`; so sánh `"abcdabcd"` với `"cdabcdab"`: khác ⇒ không chứa.
- Lặp 3 lần: `"abcdabcdabcd"` — dài 12; `b` bắt đầu ở chỉ số 2 (`"cdabcdab"`) ⇒ chứa. Đáp án **3**.

Vì sao phải lên 3 dù `8/4 = 2`? Vì `b` **không bắt đầu đúng ở đầu một bản sao** của `a`: nó bắt đầu ở chỉ số `2` (giữa bản sao thứ nhất), nên "tốn" thêm một phần của bản sao thứ ba.

**Ví dụ 2:** `a = "a"`, `b = "aa"`: lặp 1 lần chỉ `"a"` (ngắn hơn); lặp 2 lần `"aa"` chứa `b` ⇒ **2**.

**Ví dụ -1:** `a = "abc"`, `b = "ac"`: `"abcabcabc..."` không bao giờ có `"ac"` liền nhau ⇒ `-1`. Và `a = "ab"`, `b = "ba"`: `"abab"` chứa `"ba"` ⇒ 2.

Quan sát chính: số lần cần lặp **tối thiểu** là `ceil(len(b) / len(a))` (phải đủ dài), và nếu `b` không bắt đầu ở đầu `a`, cần **thêm tối đa một** bản nữa.

## 5. Cách 1 — Brute Force / baseline: thử lặp `1, 2, 3, ...` đến một ngưỡng rộng

**Ý tưởng:** với `k = 1, 2, 3, ...`, dựng `s = a * k` và kiểm tra `b in s`; trả `k` đầu tiên thoả, vì ta thử theo thứ tự tăng nên là **nhỏ nhất**. Nhưng phải dừng ở đâu? Nếu dừng quá sớm ta sai, nếu không dừng thì **lặp vô hạn** khi đáp án là `-1`. Với baseline, ta chọn một **ngưỡng rộng rãi** không dựa vào chứng minh nào: `k` tới `len(b) // len(a) + 5` — cố ý "dư" để làm **oracle** kiểm tra lập luận ở mục 6.

In [7]:
def rsm_bruteforce(a, b):
    """Cách 1: thử k = 1, 2, ... tới ngưỡng rộng (oracle, không dựa vào chứng minh chặn trên)."""
    limit = len(b) // len(a) + 5
    for k in range(1, limit + 1):
        if b in a * k:
            return k
    return -1


for a, b, expected in examples:
    print(a, b, "->", rsm_bruteforce(a, b), "(mong đợi", expected, ")")
    assert rsm_bruteforce(a, b) == expected
print(rsm_bruteforce("abc", "ac"), rsm_bruteforce("ab", "ba"), rsm_bruteforce("abc", "wxyz"))   # -1 2 -1

abcd cdabcdab -> 3 (mong đợi 3 )
a aa -> 2 (mong đợi 2 )
-1 2 -1


**Độ phức tạp:** ngưỡng `k ≈ len(b)/len(a) + 5`; mỗi lần dựng chuỗi dài tới `k·len(a) ≈ len(b) + 5·len(a)` và tìm chuỗi con (tệ nhất `O(|s|·|b|)` với tìm kiếm ngây thơ; CPython dùng thuật toán tối ưu hơn nên thực tế nhanh). Tổng **`O((|b|/|a|+5) · (|b| + 5|a|))`** thời gian, bộ nhớ `O(|b| + |a|)`.

**Nút thắt:** (1) ta thử **nhiều lần không cần thiết** (cả những `k` quá lớn); (2) ngưỡng "5" là tuỳ tiện — ta cần biết **chặn trên chính xác**. Mục 6 chứng minh chỉ cần thử đúng **hai** giá trị `k`.

## 6. Ý tưởng thuật toán / bước đột phá

Giả sử `b` xuất hiện trong `a * k` bắt đầu ở chỉ số `p` (đếm trong chuỗi lặp).

**Bước 1 — trượt về bản sao đầu tiên.** Nếu `p ≥ len(a)`, bỏ bớt các bản sao nguyên ở đầu: `b` vẫn xuất hiện, bắt đầu ở `s = p mod len(a)`, với `k` nhỏ hơn. Vì ta tìm `k` **nhỏ nhất**, ta chỉ cần xét `s ∈ [0, len(a) - 1]`.

**Bước 2 — số lần cần để chứa `b` bắt đầu ở `s`.** `b` chiếm các chỉ số `s .. s + len(b) - 1`, nên cần chuỗi lặp dài tối thiểu `s + len(b)` ⇒ số bản sao `k = ceil((s + len(b)) / len(a))`.

**Bước 3 — chặn hai đầu.** Vì `0 ≤ s ≤ len(a) - 1`:

- `k ≥ ceil(len(b) / len(a)) =: q` (ứng với `s = 0`),
- `k ≤ ceil((len(a) - 1 + len(b)) / len(a)) ≤ q + 1`.

Vậy **nếu đáp án tồn tại thì nó là `q` hoặc `q + 1`**. Chỉ cần thử đúng hai giá trị này!

(Lời giải chính thức thử `q` rồi thêm tối đa **3** bản nữa cho an toàn; chứng minh trên cho thấy thêm **1** là đủ.)

**Hệ quả:** thuật toán gọn: `q = ceil(|b|/|a|)`; nếu `b in a*q` trả `q`; elif `b in a*(q+1)` trả `q+1`; else `-1`.

Ý tưởng khác (không dựng chuỗi lặp): thử từng điểm bắt đầu `s ∈ [0, |a|)` và so sánh `b[t]` với `a[(s + t) % |a|]` — Cách 3. Và để tránh quét lại, dùng KMP trên chuỗi lặp "ảo" — Cách 4.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: thử `k` tới ngưỡng rộng (mục 5, oracle).
- Cách 2: chỉ thử `q` và `q + 1` (**chính thức**).
- Cách 3: không dựng chuỗi — thử từng điểm bắt đầu với chỉ số vòng `% len(a)`.
- Cách 4: KMP trên chuỗi lặp ảo (tuyến tính).

---

### Cách 2 — Chỉ thử `q` và `q + 1`

Giải thích từng đoạn:

1. `q = -(-len(b) // len(a))`: số bản sao tối thiểu để chuỗi lặp **dài ít nhất** `len(b)`.
2. `s = a * q`. Nếu `b in s` ⇒ trả `q`.
3. `s += a` (thêm một bản) ⇒ `b in s` ⇒ trả `q + 1`.
4. Ngược lại `-1`.

In [8]:
def rsm_two_tries(a, b):
    """Cách 2: chỉ thử q = ceil(|b|/|a|) và q + 1 bản sao."""
    q = -(-len(b) // len(a))
    s = a * q
    if b in s:
        return q
    s += a
    if b in s:
        return q + 1
    return -1


for a, b, expected in examples:
    print(a, b, "->", rsm_two_tries(a, b))
    assert rsm_two_tries(a, b) == expected

abcd cdabcdab -> 3
a aa -> 2


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        q = -(-len(b) // len(a))
        s = a * q
        if b in s:
            return q
        s += a
        if b in s:
            return q + 1
        return -1
```

**Độ phức tạp:** dựng chuỗi dài `O(|b| + |a|)`, tìm chuỗi con bằng `in` (CPython dùng thuật toán two-way/Boyer–Moore–Horspool cải tiến, thực tế gần tuyến tính; lý thuyết tệ nhất `O(|s|·|b|)` với tìm kiếm ngây thơ). Bộ nhớ `O(|b| + |a|)`.

---

### Cách 3 — Không dựng chuỗi: thử từng điểm bắt đầu với chỉ số vòng

Giải thích từng đoạn:

1. Với mỗi điểm bắt đầu `s` từ `0` đến `len(a) - 1` (chỉ cần trong bản sao đầu, theo mục 6).
2. So sánh `b[t] == a[(s + t) % len(a)]` cho mọi `t = 0 .. len(b) - 1`. Nếu lệch ⇒ thử `s` kế tiếp.
3. Nếu khớp hết ⇒ số bản sao cần = `ceil((s + len(b)) / len(a))`; vì thử `s` **tăng dần** nên `s` hợp lệ đầu tiên cho `k` nhỏ nhất (k tăng theo `s`).
4. Không có `s` nào ⇒ `-1`.

In [9]:
def rsm_offsets(a, b):
    """Cách 3: thử từng điểm bắt đầu s trong [0, |a|), so khớp bằng chỉ số vòng; không dựng chuỗi lặp."""
    n, m = len(a), len(b)
    for s in range(n):
        for t in range(m):
            if b[t] != a[(s + t) % n]:
                break
        else:                                    # khớp trọn vẹn
            return -(-(s + m) // n)             # ceil((s + m) / n)
    return -1


for a, b, expected in examples:
    print(a, b, "->", rsm_offsets(a, b))
    assert rsm_offsets(a, b) == expected
print(rsm_offsets("abc", "ac"), rsm_offsets("ab", "ba"), rsm_offsets("abc", "cabcabca"))   # -1 2 4

abcd cdabcdab -> 3
a aa -> 2
-1 2 4


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        n, m = len(a), len(b)
        for s in range(n):
            for t in range(m):
                if b[t] != a[(s + t) % n]:
                    break
            else:
                return -(-(s + m) // n)
        return -1
```

**Độ phức tạp:** `O(|a| · |b|)` thời gian tệ nhất (`10^8` khi `|a| = |b| = 10^4` với dữ liệu xấu như `a = "aaa...ab"`, `b = "aaa...a"+...`) — **quá chậm** trong Python; bộ nhớ `O(1)`. Ưu điểm: không tốn RAM dựng chuỗi, ý tưởng rõ ràng; nhược điểm: quét lại nhiều.

---

### Cách 4 — KMP trên chuỗi lặp "ảo"

Giải thích từng đoạn:

1. Tính `lps = prefix_function(b)` (mục 3.4) — `O(|b|)`.
2. Quét "văn bản" là chuỗi lặp ảo: vị trí `p = 0, 1, 2, ...`, ký tự `a[p % n]`. Chỉ cần quét tới độ dài `n · (q + 1)` (đủ cho `q + 1` bản sao).
3. Duy trì `j` = số ký tự của `b` đang khớp. Với mỗi ký tự `ch`: trong khi `j > 0` và `ch != b[j]` thì `j = lps[j - 1]`; nếu `ch == b[j]` thì `j += 1`.
4. Khi `j == m` (đã khớp trọn `b`), kết thúc tại vị trí `p`, tức đã dùng `p + 1` ký tự ⇒ số bản sao `ceil((p + 1) / n)`. Vì quét tăng dần nên **lần khớp đầu tiên cho `k` nhỏ nhất**.
5. Quét hết mà không khớp ⇒ `-1`.

In [10]:
def rsm_kmp(a, b):
    """Cách 4: KMP trên chuỗi lặp ảo a a a ... (chỉ quét tối đa n*(q+1) ký tự)."""
    n, m = len(a), len(b)
    lps = prefix_function(b)
    q = -(-m // n)
    j = 0
    for p in range(n * (q + 1)):
        ch = a[p % n]
        while j > 0 and ch != b[j]:
            j = lps[j - 1]
        if ch == b[j]:
            j += 1
        if j == m:                               # khớp xong b, kết thúc ở ký tự thứ p (đếm từ 0)
            return -(-(p + 1) // n)
    return -1


for a, b, expected in examples:
    print(a, b, "->", rsm_kmp(a, b))
    assert rsm_kmp(a, b) == expected
print(rsm_kmp("abc", "ac"), rsm_kmp("ab", "ba"), rsm_kmp("abc", "cabcabca"))   # -1 2 4

abcd cdabcdab -> 3
a aa -> 2
-1 2 4


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        n, m = len(a), len(b)
        lps = [0] * m
        k = 0
        for i in range(1, m):
            while k > 0 and b[i] != b[k]:
                k = lps[k - 1]
            if b[i] == b[k]:
                k += 1
            lps[i] = k
        q = -(-m // n)
        j = 0
        for p in range(n * (q + 1)):
            ch = a[p % n]
            while j > 0 and ch != b[j]:
                j = lps[j - 1]
            if ch == b[j]:
                j += 1
            if j == m:
                return -(-(p + 1) // n)
        return -1
```

**Độ phức tạp:** thời gian **`O(|a| + |b|)`** chắc chắn (KMP đảm bảo mỗi ký tự văn bản làm `j` tăng tối đa 1 và các lần lùi tổng không vượt số lần tăng); độ dài văn bản quét `n(q+1) ≤ |b| + 2|a|`; bộ nhớ `O(|b|)` cho `lps`. Đây là cách có đảm bảo lý thuyết tốt nhất, hữu ích khi `in` của ngôn ngữ không tối ưu.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 với `a = "abcd"`, `b = "cdabcdab"`

`n = 4`, `m = 8`. Thử từng điểm bắt đầu `s`:

| s | so khớp (`b[t]` vs `a[(s+t)%4]`) | kết quả |
|---|---|---|
| 0 | `c` vs `a` | lệch ngay ở `t = 0` |
| 1 | `c` vs `b` | lệch ngay |
| 2 | `c`/`c`, `d`/`d`, `a`/`a`, `b`/`b`, `c`/`c`, `d`/`d`, `a`/`a`, `b`/`b` | **khớp trọn** |

Khi `s = 2`: `k = ceil((2 + 8) / 4) = ceil(2.5) = 3` ✓ (khớp với ví dụ 1). Hàm trace in từng bước.

In [11]:
def trace_offsets(a, b):
    n, m = len(a), len(b)
    print(f"n={n}, m={m}")
    for s in range(n):
        matched = 0
        for t in range(m):
            x, y = b[t], a[(s + t) % n]
            if x != y:
                print(f"s={s}: t={t}: b[{t}]={x!r} != a[{(s + t) % n}]={y!r} -> lệch (đã khớp {matched})")
                break
            matched += 1
        else:
            k = -(-(s + m) // n)
            print(f"s={s}: khớp cả {m} ký tự -> k = ceil(({s}+{m})/{n}) = {k}")
            return k
    print("không điểm bắt đầu nào khớp -> -1")
    return -1


assert trace_offsets("abcd", "cdabcdab") == 3
print("----")
assert trace_offsets("abc", "ac") == -1

n=4, m=8
s=0: t=0: b[0]='c' != a[0]='a' -> lệch (đã khớp 0)
s=1: t=0: b[0]='c' != a[1]='b' -> lệch (đã khớp 0)
s=2: khớp cả 8 ký tự -> k = ceil((2+8)/4) = 3
----
n=3, m=2
s=0: t=1: b[1]='c' != a[1]='b' -> lệch (đã khớp 1)
s=1: t=0: b[0]='a' != a[1]='b' -> lệch (đã khớp 0)
s=2: t=0: b[0]='a' != a[2]='c' -> lệch (đã khớp 0)
không điểm bắt đầu nào khớp -> -1


### 8.2. Trace Cách 4 (KMP) với `a = "abcd"`, `b = "cdabcdab"`

`lps(b) = [0, 0, 0, 0, 1, 2, 3, 4]` (tiền tố `"cdab"` trùng hậu tố `"cdab"` ở cuối). Văn bản ảo: `abcdabcdabcd...`, quét tới `n(q+1) = 4·3 = 12` ký tự.

| p | ký tự | j trước | thao tác | j sau |
|---|---|---|---|---|
| 0 | a | 0 | `a != b[0]='c'`, `j = 0` | 0 |
| 1 | b | 0 | lệch | 0 |
| 2 | c | 0 | khớp `b[0]` | 1 |
| 3 | d | 1 | khớp `b[1]` | 2 |
| 4 | a | 2 | khớp `b[2]` | 3 |
| 5 | b | 3 | khớp `b[3]` | 4 |
| 6 | c | 4 | khớp `b[4]` | 5 |
| 7 | d | 5 | khớp `b[5]` | 6 |
| 8 | a | 6 | khớp `b[6]` | 7 |
| 9 | b | 7 | khớp `b[7]` ⇒ `j = 8 = m` | 8 |

Khớp xong tại `p = 9` ⇒ đã dùng `10` ký tự ⇒ `k = ceil(10/4) = 3` ✓.

In [12]:
def trace_kmp(a, b):
    n, m = len(a), len(b)
    lps = prefix_function(b)
    print("lps =", lps)
    q = -(-m // n)
    j = 0
    for p in range(n * (q + 1)):
        ch = a[p % n]
        before = j
        while j > 0 and ch != b[j]:
            j = lps[j - 1]
        if ch == b[j]:
            j += 1
        print(f"p={p:2d} ch={ch!r} j: {before} -> {j}")
        if j == m:
            k = -(-(p + 1) // n)
            print(f"khớp xong tại p={p}, dùng {p + 1} ký tự -> k={k}")
            return k
    print("hết văn bản ảo -> -1")
    return -1


assert trace_kmp("abcd", "cdabcdab") == 3

lps = [0, 0, 0, 0, 1, 2, 3, 4]
p= 0 ch='a' j: 0 -> 0
p= 1 ch='b' j: 0 -> 0
p= 2 ch='c' j: 0 -> 1
p= 3 ch='d' j: 1 -> 2
p= 4 ch='a' j: 2 -> 3
p= 5 ch='b' j: 3 -> 4
p= 6 ch='c' j: 4 -> 5
p= 7 ch='d' j: 5 -> 6
p= 8 ch='a' j: 6 -> 7
p= 9 ch='b' j: 7 -> 8
khớp xong tại p=9, dùng 10 ký tự -> k=3


## 9. Code nộp LeetCode

Chọn Cách 2 (ngắn gọn, đúng, dùng `in` đã tối ưu của Python): đúng `class Solution`, phương thức `repeatedStringMatch(self, a: str, b: str) -> int`.

In [13]:
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        q = -(-len(b) // len(a))        # ceil(len(b) / len(a))
        s = a * q
        if b in s:
            return q
        s += a
        if b in s:
            return q + 1
        return -1


sol = Solution()
print(sol.repeatedStringMatch("abcd", "cdabcdab"))   # 3
print(sol.repeatedStringMatch("a", "aa"))            # 2

3
2


## 10. Vì sao đúng

Gọi `n = |a|`, `m = |b|`, `q = ceil(m / n)`.

1. **Chuỗi lặp ảo:** ký tự thứ `p` của `a * k` là `a[p mod n]` (với `p < k·n`). `b` là chuỗi con của `a * k` ⇔ tồn tại `p` với `p + m ≤ k·n` và `b[t] = a[(p + t) mod n]` với mọi `t`.
2. **Chuẩn hoá điểm bắt đầu:** nếu `b` khớp tại `p`, thì cũng khớp tại `s = p mod n` (vì `a[(p + t) mod n] = a[(s + t) mod n]`). Số bản sao cần để chứa nó bắt đầu ở `s` là `ceil((s + m)/n)` ≤ số cần để chứa nó bắt đầu ở `p`. Vậy **đáp án nhỏ nhất** dùng điểm bắt đầu `s ∈ [0, n-1]` nhỏ nhất thoả khớp.
3. **Chặn:** `ceil(m/n) ≤ ceil((s + m)/n) ≤ ceil((n - 1 + m)/n) ≤ q + 1`. Vậy đáp án (nếu có) ∈ {`q`, `q + 1`}. Cách 2 thử đúng hai giá trị theo thứ tự tăng ⇒ trả giá trị nhỏ nhất.
4. **`-1` đúng:** nếu không có điểm bắt đầu `s` nào khớp thì không có `k` nào làm `b` xuất hiện (mọi lần xuất hiện đều quy về một `s` như bước 2). Mà nếu có `s`, `k ≤ q + 1` nên Cách 2 sẽ tìm ra ⇒ khi cả hai lần thử thất bại, không tồn tại đáp án.
5. **Cách 3** đúng vì liệt kê đúng mọi `s ∈ [0, n-1]` và tính `k` theo công thức ở bước 2; **Cách 4** là Cách 3 nhưng dùng KMP để so khớp nhanh, kết thúc tại lần khớp sớm nhất (kết thúc sớm nhất ⇒ `k` nhỏ nhất vì `k = ceil(vị trí kết thúc / n)` đơn điệu).
6. **Cách 1** (oracle) thử ngưỡng rộng hơn nhiều chặn trên, nên thực nghiệm ở mục 12 kiểm chứng gián tiếp lập luận chặn.

## 11. Phân tích Time / Space Complexity

`n = |a|`, `m = |b|`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử `k` tới ngưỡng rộng | `O((m/n + 5)·(m + 5n))` | `O(m + n)` | oracle |
| 2. Thử `q`, `q+1` | `O(m + n)` dựng + `in` | `O(m + n)` | **nộp bài** |
| 3. Thử từng điểm bắt đầu | `O(n · m)` tệ nhất | `O(1)` | chậm với dữ liệu xấu |
| 4. KMP trên chuỗi ảo | `O(n + m)` | `O(m)` | tuyến tính chắc chắn |

Chuỗi dựng ở Cách 2 dài `(q + 1)·n ≤ m + 2n ≤ 3·10^4` — rất nhỏ.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (cả 4 cách + `Solution`).
2. **Edge case:** `b` ngắn hơn `a`, `b == a`, `a` một ký tự, `b` vắt qua ranh giới, `b` dài hơn nhiều, chữ cái không có trong `a`, đủ chữ cái nhưng sai thứ tự, tình huống phải dùng `q + 1`.
3. **Phản ví dụ** cho các cách làm sai (mục 13).
4. **Đối chiếu nhiều input**: oracle Cách 1 với các cách còn lại trên hàng nghìn input nhỏ có seed (có cả input **chắc chắn có đáp án** — `b` cắt từ chuỗi lặp — và input ngẫu nhiên); thêm input đúng ràng buộc lớn.
5. **Hiệu năng:** `|a| = |b| = 10^4` (các ca xấu).

In [14]:
IMPLS = [rsm_bruteforce, rsm_two_tries, rsm_offsets, rsm_kmp, lambda a, b: Solution().repeatedStringMatch(a, b)]
NAMES = ["bruteforce", "two_tries", "offsets", "kmp", "Solution"]

cases = [
    ("abcd", "cdabcdab", 3),
    ("a", "aa", 2),
    ("abc", "abc", 1),                  # b == a
    ("abcd", "bc", 1),                  # b nằm trọn trong a
    ("abc", "ca", 2),                   # b vắt qua ranh giới (q = 1, cần q + 1)
    ("abc", "cabcabca", 4),             # q = 3, cần q + 1
    ("abc", "ac", -1),                  # đủ chữ cái nhưng sai thứ tự
    ("abc", "wxyz", -1),                # chữ cái không có trong a
    ("a", "a", 1),
    ("a", "b", -1),
    ("ab", "ba", 2),
    ("ab", "abababab", 4),
    ("aa", "a", 1),
    ("abcabcd", "abcdabcabc", 2),
    ("baa", "abaab", 3),
    ("a" * 5, "a" * 11, 3),            # q = ceil(11/5) = 3 và đủ
]
for a, b, expected in cases:
    for f, name in zip(IMPLS, NAMES):
        got = f(a, b)
        assert got == expected, (name, a, b, got, expected)
print(len(cases), "ca cố định: cả 5 cách đều đúng")

16 ca cố định: cả 5 cách đều đúng


In [15]:
# Đối chiếu ngẫu nhiên input NHỎ (có seed): oracle Cách 1 vs các cách khác
rng = random.Random(686)
pos = neg = 0
for _ in range(6000):
    alpha = "abc"[: rng.randint(1, 3)]
    a = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 6)))
    if rng.random() < 0.6:                       # b cắt từ chuỗi lặp -> chắc chắn có đáp án
        rep = a * (rng.randint(2, 6))
        s = rng.randint(0, len(a) - 1)
        e = rng.randint(s + 1, len(rep))
        b = rep[s:e]
    else:                                         # b hoàn toàn ngẫu nhiên
        b = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 14)))
    expected = rsm_bruteforce(a, b)
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        assert f(a, b) == expected, (name, a, b, f(a, b), expected)
    if expected == -1:
        neg += 1
    else:
        pos += 1
print(f"6000 input nhỏ ngẫu nhiên (có đáp án: {pos}, -1: {neg}): mọi cách khớp oracle")

# Kiểm chứng chặn q / q+1: đáp án luôn thuộc {q, q+1} hoặc -1
for _ in range(3000):
    a = "".join(rng.choice("ab") for _ in range(rng.randint(1, 7)))
    b = "".join(rng.choice("ab") for _ in range(rng.randint(1, 20)))
    r = rsm_bruteforce(a, b)
    q = -(-len(b) // len(a))
    assert r in (-1, q, q + 1), (a, b, r, q)
print("chặn trên q / q+1: OK (3000 input)")

6000 input nhỏ ngẫu nhiên (có đáp án: 4557, -1: 1443): mọi cách khớp oracle


chặn trên q / q+1: OK (3000 input)


In [16]:
# Input LỚN đúng ràng buộc (|a|, |b| <= 10^4): so Cách 2, 4 và Solution (Cách 3 chỉ trên cỡ vừa)
rng = random.Random(1)
for _ in range(60):
    na = rng.randint(1, 10 ** 4)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.choice([1, 2, 3, 26])]
    a = "".join(rng.choice(alpha) for _ in range(na))
    if rng.random() < 0.7:
        rep = a * (10 ** 4 // na + 3)
        s = rng.randint(0, na - 1)
        L = rng.randint(1, min(10 ** 4, len(rep) - s))
        b = rep[s:s + L]
    else:
        b = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 10 ** 4)))
    r2, r4, rs = rsm_two_tries(a, b), rsm_kmp(a, b), Solution().repeatedStringMatch(a, b)
    assert r2 == r4 == rs, (len(a), len(b), r2, r4)
print("60 input lớn: Cách 2 = Cách 4 = Solution")

# Cỡ vừa: thêm Cách 3
for _ in range(100):
    a = "".join(rng.choice("ab") for _ in range(rng.randint(1, 60)))
    rep = a * 10
    s = rng.randint(0, len(a) - 1)
    b = rep[s:s + rng.randint(1, 300)] if rng.random() < 0.7 else "".join(rng.choice("ab") for _ in range(rng.randint(1, 100)))
    assert rsm_offsets(a, b) == rsm_two_tries(a, b) == rsm_kmp(a, b)
print("100 input cỡ vừa: cả Cách 3 khớp")

60 input lớn: Cách 2 = Cách 4 = Solution
100 input cỡ vừa: cả Cách 3 khớp


In [17]:
# Hiệu năng ca xấu
n = 10 ** 4
bad = {
    "a=a...a, b=a...a (b dài)": ("a" * n, "a" * n),
    "a=a...ab, b=a...a": ("a" * (n - 1) + "b", "a" * (n - 1)),
    "a=ab, b=(ab)^k": ("ab", "ab" * (n // 2)),
    "-1: a=a..ab, b=a..ab+1": ("a" * (n - 1) + "b", "a" * (n - 2) + "bb"),
}
for label, (a, b) in bad.items():
    row = []
    for f, name in [(rsm_two_tries, "two_tries"), (rsm_kmp, "kmp"), (IMPLS[4], "Solution")]:
        t0 = time.perf_counter()
        r = f(a, b)
        row.append((name, r, time.perf_counter() - t0))
    assert len({r for _, r, _ in row}) == 1
    print(f"{label:<28} -> {row[0][1]:>3}  " + "  ".join(f"{nm}: {t:.4f}s" for nm, _, t in row))

# Cách 3 tăng theo bình phương (cỡ nhỏ hơn để không chờ lâu)
for size in (200, 400, 800):
    a, b = "a" * (size - 1) + "b", "a" * (size - 1) + "c"
    t0 = time.perf_counter()
    r = rsm_offsets(a, b)
    print(f"Cách 3, |a|=|b|={size}: kết quả {r}, {time.perf_counter() - t0:.4f}s (gấp đôi cỡ ~ gấp 4 thời gian)")

a=a...a, b=a...a (b dài)     ->   1  two_tries: 0.0000s  kmp: 0.0030s  Solution: 0.0001s
a=a...ab, b=a...a            ->   1  two_tries: 0.0000s  kmp: 0.0028s  Solution: 0.0000s
a=ab, b=(ab)^k               -> 5000  two_tries: 0.0000s  kmp: 0.0027s  Solution: 0.0000s
-1: a=a..ab, b=a..ab+1       ->  -1  two_tries: 0.0001s  kmp: 0.0055s  Solution: 0.0001s
Cách 3, |a|=|b|=200: kết quả -1, 0.0012s (gấp đôi cỡ ~ gấp 4 thời gian)
Cách 3, |a|=|b|=400: kết quả -1, 0.0063s (gấp đôi cỡ ~ gấp 4 thời gian)
Cách 3, |a|=|b|=800: kết quả -1, 0.0298s (gấp đôi cỡ ~ gấp 4 thời gian)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng chia nguyên (`//`, làm tròn xuống) thay vì chia lấy trần

`len(b) // len(a)` cho số bản sao **quá ít** khi `len(b)` không chia hết `len(a)`: chuỗi lặp ngắn hơn `b`, không thể chứa.

In [18]:
def wrong_floor(a, b):
    q = len(b) // len(a)                 # SAI: làm tròn xuống
    s = a * q
    if b in s:
        return q
    s += a
    if b in s:
        return q + 1
    return -1


print(wrong_floor("abc", "cabcabca"), "(đúng: 4)")
print(wrong_floor("abc", "ca"), "(đúng: 2)")
print(wrong_floor("abcd", "cdabcdab"), "(đúng: 3)")

-1 (đúng: 4)
-1 (đúng: 2)
3 (đúng: 3)


**Vì sao sai:** ví dụ `a = "abc"`, `b = "cabcabca"` (dài 8): `8 // 3 = 2`, thử 2 và 3 bản sao nhưng cần 4. **Sửa:** `q = -(-len(b) // len(a))` (chia trần).

### Lỗi 2: chỉ thử `q` bản sao, quên bản "dư thêm một lần"

Lập luận "đủ dài" `q·n ≥ m` chưa đủ: `b` có thể **bắt đầu giữa một bản sao** và vắt sang bản kế tiếp, cần thêm một bản.

In [19]:
def wrong_only_q(a, b):
    q = -(-len(b) // len(a))
    return q if b in a * q else -1      # SAI: thiếu thử q + 1


print(wrong_only_q("abcd", "cdabcdab"), "(đúng: 3)")
print(wrong_only_q("abc", "ca"), "(đúng: 2)")
print(wrong_only_q("a", "aa"), "(đúng: 2)")

-1 (đúng: 3)
-1 (đúng: 2)
2 (đúng: 2)


**Sửa:** thử thêm `q + 1` (mục 6).

### Lỗi 3: vòng lặp không có điều kiện dừng

`while b not in s: s += a; k += 1` ổn khi có đáp án, nhưng khi đáp án là `-1` thì **không bao giờ dừng**. Ta chạy thử với "van an toàn" (giới hạn số vòng) để thấy:

In [20]:
def wrong_infinite(a, b, guard=50):
    s, k = a, 1
    while b not in s:                      # SAI: không có điều kiện dừng thật sự
        s += a
        k += 1
        if k > guard:                      # van an toàn chỉ để notebook không treo
            return "VÒNG LẶP VÔ HẠN (bị van an toàn cắt)"
    return k


print(wrong_infinite("abcd", "cdabcdab"))
print(wrong_infinite("abc", "ac"))

3
VÒNG LẶP VÔ HẠN (bị van an toàn cắt)


**Sửa:** dừng khi `len(s) >= len(b) + len(a)` (đã chứng minh `q + 1` bản là đủ) rồi trả `-1`.

### Lỗi 4: kết luận "đủ chữ cái thì có đáp án"

Kiểm tra `set(b) <= set(a)` chỉ loại được vài ca `-1` hiển nhiên. Nếu dùng nó để **khẳng định** có đáp án rồi trả `q` hay `q+1` mà không kiểm tra chuỗi con, kết quả sai.

In [21]:
def wrong_by_set(a, b):
    if not set(b) <= set(a):
        return -1
    return -(-len(b) // len(a)) + (0 if b in a * (-(-len(b) // len(a))) else 1)     # SAI: luôn "cứu" bằng q + 1


print(wrong_by_set("abc", "ac"), "(đúng: -1)")
print(wrong_by_set("abc", "wxyz"), "(đúng: -1)")
print(wrong_by_set("abcd", "cdabcdab"), "(đúng: 3)")

2 (đúng: -1)
-1 (đúng: -1)
3 (đúng: 3)


**Sửa:** sau khi thử `q` và `q + 1` mà vẫn không có `b` thì mới trả `-1`.

### Lỗi 5: đếm sai số bản sao khi dựng bằng vòng lặp (lệch 1)

Khởi tạo `s = ""`, `k = 0` rồi `while len(s) < len(b): s += a; k += 1` đúng; nhưng nếu khởi tạo `s = a` mà `k = 0` thì đáp án **thiếu 1**. Luôn kiểm tra **bất biến**: "`s` là `a` lặp `k` lần".

In [22]:
def wrong_off_by_one(a, b):
    s, k = a, 0                            # SAI: s đã là 1 bản mà k = 0
    while len(s) < len(b):
        s += a
        k += 1
    if b in s:
        return k
    s += a
    k += 1
    return k if b in s else -1


print(wrong_off_by_one("abcd", "cdabcdab"), "(đúng: 3)")
print(wrong_off_by_one("a", "aa"), "(đúng: 2)")

2 (đúng: 3)
1 (đúng: 2)


**Checklist chống lỗi:**

- [ ] Chia **trần** cho `q`.
- [ ] Thử **cả** `q` và `q + 1`; sau đó mới `-1`.
- [ ] Không có vòng lặp không dừng.
- [ ] `set(b) <= set(a)` chỉ là điều kiện cần.
- [ ] Bất biến "chuỗi đang dựng = `a` lặp đúng `k` lần".

## 14. Mẹo & nhận dạng pattern

- **"Lặp chuỗi rồi tìm chuỗi con"** ⇒ nghĩ ngay **chặn số lần lặp**: sau khi chuẩn hoá điểm bắt đầu về bản sao đầu, chỉ cần `ceil(m/n)` hoặc `+1`.
- **Chuỗi tuần hoàn / vòng** ⇒ nhân đôi chuỗi (`a + a`) hoặc dùng chỉ số `% n` thay vì dựng chuỗi dài. Bài 796 (Rotate String): `b` là phép xoay của `a` ⇔ `len(a) == len(b)` và `b in a + a`.
- **Tìm chuỗi con tuyến tính** ⇒ KMP (mảng `lps`), Z-algorithm, hoặc băm cuộn (Rabin–Karp). Trong Python thường chỉ cần `in` / `find`.
- **Chia lấy trần** `-(-x // y)` là mẫu số nguyên dùng rất nhiều (số trang, số lần, số nhóm).
- **Tính nhỏ nhất** thường đi cùng "thử theo thứ tự tăng và trả cái đầu tiên".
- **Nhớ chặn trên** để vòng lặp luôn dừng; "đáp án -1" thường là kết thúc của vòng lặp có giới hạn.
- Bài họ hàng: 28 (Find the Index of the First Occurrence), 459 (Repeated Substring Pattern), 796 (Rotate String), 1392 (Longest Happy Prefix), 214 (Shortest Palindrome).

## 15. Biến thể

1. **Bài 459 — Repeated Substring Pattern:** `s` có phải là một chuỗi con lặp lại nhiều lần không? Mẹo: `s in (s + s)[1:-1]`.
2. **Bài 796 — Rotate String:** `a` xoay thành `b` ⇔ `len(a) == len(b) and b in a + a`.
3. **Trả về cả vị trí bắt đầu `s`** (không chỉ số lần lặp): từ Cách 3/4.
4. **Tối đa hoá thay vì tối thiểu:** số lần lặp tối đa mà `b` vẫn là chuỗi con? — không giới hạn trên (vô hạn), nên đề thường đổi thành "đếm số lần `b` xuất hiện".
5. **Dãy con (subsequence) thay cho chuỗi con:** `b` là dãy con của `a` lặp `k` lần — dùng hai con trỏ trên chuỗi lặp; số lần = số vòng qua `a`.

Minh hoạ 1 và 2:

In [23]:
def repeated_substring_pattern(s):
    """Bài 459: s có thể tạo bằng cách lặp một chuỗi con >= 2 lần?"""
    return s in (s + s)[1:-1]


def rotate_string(a, b):
    """Bài 796."""
    return len(a) == len(b) and b in a + a


print(repeated_substring_pattern("abab"), repeated_substring_pattern("aba"), repeated_substring_pattern("abcabcabc"))   # True False True
print(rotate_string("abcde", "cdeab"), rotate_string("abcde", "abced"))                                                # True False

True False True
True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `k` tới ngưỡng rộng | lặp tăng dần, kiểm `b in a*k` | `O((m/n+5)(m+5n))` | `O(m+n)` | oracle/học |
| 2. Thử `q`, `q+1` | chặn trên bằng chứng minh | `O(m+n)` (với `in` tốt) | `O(m+n)` | **nộp bài** |
| 3. Thử từng điểm bắt đầu | chỉ số vòng `%`, không dựng chuỗi | `O(nm)` tệ nhất | `O(1)` | `n`, `m` nhỏ; muốn tiết kiệm RAM |
| 4. KMP trên chuỗi ảo | `lps` + quét văn bản ảo | `O(n+m)` | `O(m)` | cần đảm bảo tuyến tính |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Dự đoán giá trị: (1) `-(-7 // 3)`; (2) `"ab" * 3`; (3) `"ca" in "abcabc"`; (4) `Solution().repeatedStringMatch("ab", "bab")`. Ghi list `bai1_du_doan`.

In [24]:
# TODO Bài 1: bai1_du_doan = [..., ..., ..., ...]
bai1_du_doan = None

### Bài 2 (tính tay)

`a = "abc"`, `b = "cabcabca"`. (a) `q = ceil(|b|/|a|)` bằng bao nhiêu? (b) Đáp án bài toán? Ghi `(q, dap_an)` vào `bai2`.

In [25]:
# TODO Bài 2: bai2 = (q, dap_an)
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành `ceil_div(x, y)` (chia lấy trần bằng số nguyên) và `repeat_match(a, b)` (Cách 2).

In [26]:
def ceil_div(x, y):
    # TODO: chia lấy trần bằng số nguyên
    pass


def repeat_match(a, b):
    # TODO: q = ceil_div(len(b), len(a)); thử q bản, rồi q + 1 bản; không có thì -1
    pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết hàm `repeated_string_match(a, b)` theo Cách 3 (thử từng điểm bắt đầu `s` bằng chỉ số vòng, không dựng chuỗi lặp).

In [27]:
# TODO Bài 4: tự viết lại
def repeated_string_match(a, b):
    pass

### Bài 5 (biến thể / mini challenge)

Bài 459: viết `repeated_substring_pattern(s)` trả `True` nếu `s` được tạo bằng cách lặp một chuỗi con **ít nhất 2 lần**. Sau đó giải thích vì sao `s in (s + s)[1:-1]` đúng.

In [28]:
# TODO Bài 5
def repeated_substring_pattern(s):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `//` làm tròn xuống; `-(-x // y)` làm tròn lên.
- Mức 2: `"ca"` nằm ở chỉ số 2 của `"abcabc"`.
- Mức 3: `[3, "ababab", True, 2]`; với `a = "ab"`, `b = "bab"`: `q = 2`, `"abab"` chứa `"bab"` ⇒ 2.

**Bài 2**
- Mức 1: `8 / 3 = 2.67`.
- Mức 2: `a * 3 = "abcabcabc"` (dài 9) có chứa `"cabcabca"` không? Nó phải bắt đầu ở chỉ số 2 và cần tới chỉ số 9.
- Mức 3: `q = 3`, cần 4 bản sao ⇒ `(3, 4)`.

**Bài 3**
- Mức 1: `ceil_div(x, y) = -(-x // y)`.
- Mức 2: `s = a * q`.
- Mức 3: `if b in s: return q`; `s += a`; `if b in s: return q + 1`; `return -1`.

**Bài 4**
- Mức 1: vòng ngoài `s` từ `0` đến `len(a) - 1`.
- Mức 2: vòng trong `t`, so `b[t]` với `a[(s + t) % n]`, `break` khi lệch; dùng `for ... else`.
- Mức 3: khi khớp: `return -(-(s + m) // n)`.

**Bài 5**
- Mức 1: nếu `s = t * k` thì `s + s` chứa `t` ở nhiều vị trí.
- Mức 2: bỏ ký tự đầu và cuối của `s + s`, rồi tìm `s`.
- Mức 3: `s in (s + s)[1:-1]`. Lý do: `s` luôn xuất hiện ở vị trí 0 và `len(s)` của `s + s`; bị cắt đi bởi `[1:-1]`. Nếu `s` còn xuất hiện ở vị trí khác thì `s` bằng chính nó dịch vòng một khoảng `< len(s)` ⇒ `s` tuần hoàn.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [29]:
# Bài 1
dap_an_1 = [-(-7 // 3), "ab" * 3, "ca" in "abcabc", Solution().repeatedStringMatch("ab", "bab")]
print(dap_an_1)
assert dap_an_1 == [3, "ababab", True, 2]

# Bài 2
q = -(-len("cabcabca") // len("abc"))
bai2_dap_an = (q, Solution().repeatedStringMatch("abc", "cabcabca"))
print(bai2_dap_an)
assert bai2_dap_an == (3, 4)

# Bài 3
def ceil_div(x, y):
    return -(-x // y)


def repeat_match(a, b):
    q = ceil_div(len(b), len(a))
    s = a * q
    if b in s:
        return q
    s += a
    if b in s:
        return q + 1
    return -1


assert ceil_div(7, 3) == 3 and ceil_div(6, 3) == 2 and ceil_div(1, 5) == 1
for a, b, expected in cases:
    assert repeat_match(a, b) == expected
print("Bài 3: OK")

[3, 'ababab', True, 2]
(3, 4)
Bài 3: OK


In [30]:
# Bài 4
def repeated_string_match(a, b):
    n, m = len(a), len(b)
    for s in range(n):
        for t in range(m):
            if b[t] != a[(s + t) % n]:
                break
        else:
            return -(-(s + m) // n)
    return -1


for a, b, expected in cases:
    assert repeated_string_match(a, b) == expected
rng = random.Random(4)
for _ in range(1500):
    a = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6)))
    b = "".join(rng.choice("ab") for _ in range(rng.randint(1, 14)))
    assert repeated_string_match(a, b) == rsm_bruteforce(a, b)
print("Bài 4: OK")

# Bài 5
def repeated_substring_pattern(s):
    return s in (s + s)[1:-1]


def rsp_oracle(s):
    n = len(s)
    return any(n % d == 0 and s[:d] * (n // d) == s for d in range(1, n))


assert repeated_substring_pattern("abab") and not repeated_substring_pattern("aba") and not repeated_substring_pattern("a")
rng = random.Random(5)
for _ in range(3000):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 12)))
    assert repeated_substring_pattern(s) == rsp_oracle(s), s
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "lặp chuỗi tìm chuỗi con":**

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str) -> int:
        q = -(-len(b) // len(a))        # ceil(len(b) / len(a)): số bản sao tối thiểu để đủ dài
        s = a * q
        if b in s:
            return q
        s += a                           # b có thể vắt qua ranh giới: thêm đúng 1 bản
        if b in s:
            return q + 1
        return -1
```

**Checklist trước khi submit:**

- [ ] `q` là chia **trần** (`-(-x // y)`), không phải chia sàn?
- [ ] Thử **cả** `q` và `q + 1`, chỉ trả `-1` sau đó?
- [ ] Hiểu chứng minh: chuẩn hoá điểm bắt đầu `s < len(a)` ⇒ `k ≤ q + 1`?
- [ ] Không có vòng lặp có thể chạy vô hạn?
- [ ] Đã test: `b == a`, `b` trong `a`, `b` vắt ranh giới, đủ chữ cái nhưng sai thứ tự (`"abc"`, `"ac"`), chữ cái lạ?
- [ ] Đã đối chiếu oracle ngưỡng rộng trên input nhỏ ngẫu nhiên có seed?
- [ ] Nếu cần đảm bảo tuyến tính: KMP trên chuỗi lặp ảo?